# Precompute PCA Embeddings for Webapp

> **Note:** This notebook was a one-time preprocessing step run to generate the pre-computed PCA
> parquet files bundled with the `9.webapp/` Streamlit app. It is **not intended to be re-run**
> without modification.
>
> The helper functions below (`load_model_data`, `single_load_data`, `latent_load_data`) were
> originally defined in `9.webapp/app_utils.py` but were removed from the app once the pre-computed
> parquets replaced runtime PCA computation. They are reproduced here for provenance.
>
> **Dependencies that are no longer in the webapp environment:**
> - `CRISPRGeneEffect.parquet` and `CRISPR_gene_dictionary.parquet` (large gitignored DepMap files
>   — download via `0.data-download/` first)
> - `sklearn` (`scikit-learn`) — not in `9.webapp/pyproject.toml`
>
> **Outputs** (written to `9.webapp/data/`):
> - `pca_embeddings_single_dependencies.parquet`
> - `pca_embeddings_latent_reactome.parquet`
> - `pca_embeddings_latent_corum.parquet`
> - `pca_embeddings_latent_drug.parquet`

In [ ]:
import pathlib

import pandas as pd
from sklearn.decomposition import PCA

In [ ]:
# Paths — run from repo root or adjust accordingly
REPO_ROOT = pathlib.Path().resolve().parent  # goes up from 5.drug-dependency/ to repo root
WEBAPP_DIR = REPO_ROOT / "9.webapp"
DATA_DIR = WEBAPP_DIR / "data"
RESULTS_DIR = REPO_ROOT / "5.drug-dependency" / "results"

print(f"Repo root:   {REPO_ROOT}")
print(f"Webapp data: {DATA_DIR}")
print(f"Results dir: {RESULTS_DIR}")

## Helper functions

Originally from `9.webapp/app_utils.py`, reproduced here for provenance.

In [ ]:
def load_model_data(dependency_file, gene_dict_file):
    """Load and preprocess gene dependency data and gene dictionary."""
    dependency_df = pd.read_parquet(dependency_file)
    print(dependency_df.shape)

    gene_dict_df = (
        pd.read_parquet(gene_dict_file).query("qc_pass").reset_index(drop=True)
    )
    gene_dict_df.entrez_id = gene_dict_df.entrez_id.astype(str)

    entrez_genes = [
        x[1].strip(")").strip()
        for x in dependency_df.iloc[:, 1:].columns.str.split("(")
    ]
    entrez_intersection = list(
        set(gene_dict_df.entrez_id).intersection(set(entrez_genes))
    )
    gene_dict_df = gene_dict_df.set_index("entrez_id").reindex(entrez_intersection)

    dependency_df.columns = ["ModelID"] + entrez_genes
    dependency_df = dependency_df.loc[:, ["ModelID"] + gene_dict_df.index.tolist()]
    dependency_df.columns = ["ModelID"] + gene_dict_df.symbol_id.tolist()
    dependency_df = dependency_df.dropna(axis="columns")
    return dependency_df, gene_dict_df

In [ ]:
def single_load_data():
    """Load CRISPR gene dependency scores merged with cell line metadata."""
    dependency_file = DATA_DIR / "CRISPRGeneEffect.parquet"
    gene_dict_file = DATA_DIR / "CRISPR_gene_dictionary.parquet"
    cancer_type_input_file = DATA_DIR / "Model.parquet"

    dependency_df, _ = load_model_data(dependency_file, gene_dict_file)
    dependency_df = dependency_df.set_index("ModelID")
    cancer_type_df = pd.read_parquet(cancer_type_input_file)
    combined_df = dependency_df.merge(
        cancer_type_df[["ModelID", "OncotreePrimaryDisease"]], on="ModelID", how="left"
    )
    return combined_df

In [ ]:
def latent_load_data():
    """Load latent score parquets and pivot to wide format for PCA."""
    reactome_df = pd.read_parquet(RESULTS_DIR / "all_reactome_results.parquet")
    corum_df = pd.read_parquet(RESULTS_DIR / "all_corum_results.parquet")
    drug_df = pd.read_parquet(RESULTS_DIR / "all_drug_results.parquet")

    reactome_df["feature"] = reactome_df["reactome_pathway"]
    reactome_df = reactome_df.drop(
        columns=["model", "latent_dim_total", "init", "seed", "z", "reactome_pathway"]
    )
    corum_df["feature"] = corum_df["reactome_pathway"]
    corum_df = corum_df.drop(
        columns=["model", "latent_dim_total", "init", "seed", "z", "reactome_pathway"]
    )
    drug_df["feature"] = drug_df["name"]
    drug_df = drug_df.drop(
        columns=["model", "latent_dim_total", "init", "seed", "z", "name"]
    )

    meta_cols = ["ModelID", "OncotreePrimaryDisease"]
    reactome_meta = reactome_df[meta_cols].drop_duplicates()
    corum_meta = corum_df[meta_cols].drop_duplicates()
    drug_meta = drug_df[meta_cols].drop_duplicates()

    reactome_matrix = reactome_df.pivot(
        index="ModelID", columns="feature", values="latent_score"
    ).merge(reactome_meta, on="ModelID", how="left")
    corum_matrix = corum_df.pivot(
        index="ModelID", columns="feature", values="latent_score"
    ).merge(corum_meta, on="ModelID", how="left")
    drug_matrix = drug_df.pivot(
        index="ModelID", columns="feature", values="latent_score"
    ).merge(drug_meta, on="ModelID", how="left")

    return reactome_matrix, corum_matrix, drug_matrix

## Compute and save PCA embeddings

In [ ]:
def compute_pca(df, metadata_columns) -> pd.DataFrame:
    print(df.shape[0], "rows before dropping NA")
    df = df.dropna()
    print(df.shape[0], "rows after dropping NA")
    metadata_df = df[metadata_columns].copy()
    feat_cols = df.columns.drop(metadata_columns)
    if "source" in feat_cols:
        feat_cols = feat_cols.drop("source")
    pca_input = df[feat_cols].apply(pd.to_numeric, errors="coerce")
    pca = PCA(n_components=2, random_state=0)
    pca_embedding = pca.fit_transform(pca_input)
    metadata_df["PCA1"] = pca_embedding[:, 0]
    metadata_df["PCA2"] = pca_embedding[:, 1]
    return metadata_df

In [ ]:
combined_df = single_load_data()
reactome_matrix, corum_matrix, drug_matrix = latent_load_data()

pca_single_df = compute_pca(combined_df, ["ModelID", "OncotreePrimaryDisease"])
pca_latent_reactome_df = compute_pca(reactome_matrix, ["OncotreePrimaryDisease", "ModelID"])
pca_latent_corum_df = compute_pca(corum_matrix, ["OncotreePrimaryDisease", "ModelID"])
pca_latent_drug_df = compute_pca(drug_matrix, ["OncotreePrimaryDisease", "ModelID"])

pca_single_df.to_parquet(DATA_DIR / "pca_embeddings_single_dependencies.parquet")
pca_latent_reactome_df.to_parquet(DATA_DIR / "pca_embeddings_latent_reactome.parquet")
pca_latent_corum_df.to_parquet(DATA_DIR / "pca_embeddings_latent_corum.parquet")
pca_latent_drug_df.to_parquet(DATA_DIR / "pca_embeddings_latent_drug.parquet")

print("Done — parquets written to", DATA_DIR)